In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader, WeightedRandomSampler

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


Using device: cuda


In [2]:
dataset_path = "/kaggle/input/datasets/aryashah2k/breast-ultrasound-images-dataset/Dataset_BUSI_with_GT"

In [13]:
from torchvision.datasets import ImageFolder
import os

class BUSIDataset(ImageFolder):

    def is_valid_file(self, path):
        return "_mask" not in path


dataset = ImageFolder(
    dataset_path,
    transform=basic_transform,
    is_valid_file=lambda path: "_mask" not in path
)

In [14]:

basic_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

augment_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224),
    transforms.ToTensor()
])

In [18]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder(
    dataset_path,
    transform=basic_transform,
    is_valid_file=lambda path: "_mask" not in path
)

In [16]:
train_idx, test_idx = train_test_split(
    list(range(len(dataset))),
    test_size=0.2,
    stratify=dataset.targets,
    random_state=42
)

train_dataset = torch.utils.data.Subset(dataset, train_idx)
test_dataset = torch.utils.data.Subset(dataset, test_idx)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


In [20]:
labels = dataset.targets

from collections import Counter
class_counts = Counter(labels)

print("Class Distribution:\n")

for i, class_name in enumerate(dataset.classes):
    print(f"{class_name}: {class_counts[i]}")

Class Distribution:

benign: 437
malignant: 210
normal: 133


In [21]:
def create_model():

    model = models.resnet18(weights=ResNet18_Weights.DEFAULT)

    model.fc = nn.Linear(model.fc.in_features, 3)

    return model.to(device)


In [22]:
def train_model(model, train_loader, criterion, epochs=10):

    optimizer = optim.Adam(model.parameters(), lr=0.001)

    for epoch in range(epochs):

        model.train()
        running_loss = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} Loss:", running_loss)

    return model

In [23]:
def evaluate_model(model, test_loader):

    model.eval()

    preds = []
    true = []

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(device)

            outputs = model(images)

            _, p = torch.max(outputs, 1)

            preds.extend(p.cpu().numpy())
            true.extend(labels.numpy())

    print("\nClassification Report\n")
    print(classification_report(true, preds, target_names=class_names))

    acc = accuracy_score(true, preds)

    return acc



In [24]:
print("\nTraining Baseline Model\n")

model = create_model()

criterion = nn.CrossEntropyLoss()

model = train_model(model, train_loader, criterion)

baseline_accuracy = evaluate_model(model, test_loader)

print("Baseline Accuracy:", baseline_accuracy)


Training Baseline Model

Epoch 1/10 Loss: 25.409418389201164
Epoch 2/10 Loss: 13.853224344551563
Epoch 3/10 Loss: 12.430428266525269
Epoch 4/10 Loss: 10.056179776787758
Epoch 5/10 Loss: 9.590187840163708
Epoch 6/10 Loss: 7.231511145830154
Epoch 7/10 Loss: 4.912367876619101
Epoch 8/10 Loss: 6.3983650375157595
Epoch 9/10 Loss: 4.689814830198884
Epoch 10/10 Loss: 4.2157153016887605

Classification Report

              precision    recall  f1-score   support

      benign       0.95      0.51      0.67       179
   malignant       0.52      0.92      0.66        84
      normal       0.69      0.92      0.79        53

    accuracy                           0.69       316
   macro avg       0.72      0.79      0.71       316
weighted avg       0.79      0.69      0.69       316

Baseline Accuracy: 0.689873417721519


In [25]:
class_sample_count = np.array([class_counts[i] for i in range(len(class_counts))])

weights = 1. / class_sample_count

samples_weight = np.array([weights[t] for t in labels])

samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

train_loader_over = DataLoader(train_dataset, batch_size=32, sampler=sampler)

print("\nTraining Oversampling Model\n")

model = create_model()

criterion = nn.CrossEntropyLoss()

model = train_model(model, train_loader_over, criterion)

oversampling_accuracy = evaluate_model(model, test_loader)

print("Oversampling Accuracy:", oversampling_accuracy)


Training Oversampling Model

Epoch 1/10 Loss: 16.4809108376503
Epoch 2/10 Loss: 8.719804018735886
Epoch 3/10 Loss: 7.88860160112381
Epoch 4/10 Loss: 7.048935744911432
Epoch 5/10 Loss: 5.649095043540001
Epoch 6/10 Loss: 5.695336565375328
Epoch 7/10 Loss: 3.8328981259837747
Epoch 8/10 Loss: 2.434919417835772
Epoch 9/10 Loss: 4.629302559420466
Epoch 10/10 Loss: 3.5675483494997025

Classification Report

              precision    recall  f1-score   support

      benign       0.92      0.79      0.85       179
   malignant       0.64      0.88      0.74        84
      normal       0.92      0.83      0.87        53

    accuracy                           0.82       316
   macro avg       0.83      0.83      0.82       316
weighted avg       0.85      0.82      0.82       316

Oversampling Accuracy: 0.819620253164557


In [26]:
dataset_aug = datasets.ImageFolder(dataset_path, transform=augment_transform)

train_dataset_aug = torch.utils.data.Subset(dataset_aug, train_idx)

train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)

print("\nTraining Augmentation Model\n")

model = create_model()

criterion = nn.CrossEntropyLoss()

model = train_model(model, train_loader_aug, criterion)

augmentation_accuracy = evaluate_model(model, test_loader)

print("Augmentation Accuracy:", augmentation_accuracy)



Training Augmentation Model

Epoch 1/10 Loss: 34.40422850847244
Epoch 2/10 Loss: 28.561964631080627
Epoch 3/10 Loss: 27.131534814834595
Epoch 4/10 Loss: 27.315509259700775
Epoch 5/10 Loss: 25.12077835202217
Epoch 6/10 Loss: 24.428129732608795
Epoch 7/10 Loss: 23.756324589252472
Epoch 8/10 Loss: 23.798914819955826
Epoch 9/10 Loss: 22.682514876127243
Epoch 10/10 Loss: 23.81660360097885

Classification Report

              precision    recall  f1-score   support

      benign       0.90      0.84      0.87       179
   malignant       0.68      0.89      0.77        84
      normal       0.92      0.68      0.78        53

    accuracy                           0.83       316
   macro avg       0.83      0.80      0.81       316
weighted avg       0.84      0.83      0.83       316

Augmentation Accuracy: 0.8259493670886076


In [27]:
class FocalLoss(nn.Module):

    def __init__(self, gamma=2):

        super(FocalLoss, self).__init__()

        self.gamma = gamma

        self.ce = nn.CrossEntropyLoss()

    def forward(self, inputs, targets):

        ce_loss = self.ce(inputs, targets)

        pt = torch.exp(-ce_loss)

        loss = (1 - pt) ** self.gamma * ce_loss

        return loss


print("\nTraining Focal Loss Model\n")

model = create_model()

criterion = FocalLoss()

model = train_model(model, train_loader, criterion)

focal_accuracy = evaluate_model(model, test_loader)

print("Focal Loss Accuracy:", focal_accuracy)



Training Focal Loss Model

Epoch 1/10 Loss: 8.234411541372538
Epoch 2/10 Loss: 1.904981980100274
Epoch 3/10 Loss: 0.7657007746165618
Epoch 4/10 Loss: 0.5778699764923658
Epoch 5/10 Loss: 0.9312220005958807
Epoch 6/10 Loss: 0.4662342492083553
Epoch 7/10 Loss: 0.17945247705210932
Epoch 8/10 Loss: 0.1275290296143794
Epoch 9/10 Loss: 0.32366656665180926
Epoch 10/10 Loss: 0.14608984325968777

Classification Report

              precision    recall  f1-score   support

      benign       0.94      0.89      0.91       179
   malignant       0.93      0.85      0.89        84
      normal       0.74      0.98      0.85        53

    accuracy                           0.89       316
   macro avg       0.87      0.90      0.88       316
weighted avg       0.90      0.89      0.89       316

Focal Loss Accuracy: 0.8924050632911392


In [28]:
results = pd.DataFrame({

"Model":[
"Baseline",
"Oversampling",
"Data Augmentation",
"Focal Loss"
],

"Accuracy":[
baseline_accuracy,
oversampling_accuracy,
augmentation_accuracy,
focal_accuracy
]

})

print("\nFinal Comparison\n")

print(results)


Final Comparison

               Model  Accuracy
0           Baseline  0.689873
1       Oversampling  0.819620
2  Data Augmentation  0.825949
3         Focal Loss  0.892405
